# Explore PROMISE12 case metadata

Goal: try to figure out which training cases came from which contributing center.

**Caveat:** the downloaded PROMISE12 files (`.mhd` headers, `training_data.csv`, and the
Zenodo `download_manifest.json`) contain **no institution/center label** for any case.
The original challenge paper (Litjens et al. 2014, PMC4137968) confirms 4 contributing
centers (RUNMC/Nijmegen, BIDMC/Boston, UCL/London, and a 4th referred to as "HK") and
states cases were stratified across centers, but does not publish a numeric
case-ID -> center table, and no other verifiable public source for that mapping was found.

So instead of asserting per-case institution labels, this notebook explores the raw
geometric header metadata (spacing, matrix size, direction/orientation) as a **proxy**
signal for distinct scanners/protocols, and looks for natural clustering. Any resulting
clusters are labeled generically (`Site A/B/C/D`) — they are *not* verified institution
assignments.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

DATA_DIR = Path("../data/raw/zenodo/promise12/record-8026660/promise12/training_data")
assert DATA_DIR.exists(), DATA_DIR

## Parse raw `.mhd` text headers

`.mhd` headers are plain `key = value` text files. sitk's `GetMetaDataKeys()` does not
expose fields like `TransformMatrix`/`Offset` for MetaImage files (they're consumed into
image geometry instead), so we parse the text directly as well, to cross-check against
sitk's geometry readout and to capture `ElementType`.

In [2]:
def parse_mhd_header(path: Path) -> dict:
    fields = {}
    for line in path.read_text().splitlines():
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        fields[key.strip()] = value.strip()
    return fields


sample_fields = parse_mhd_header(DATA_DIR / "Case00.mhd")
sample_fields

{'ObjectType': 'Image',
 'NDims': '3',
 'BinaryData': 'True',
 'BinaryDataByteOrderMSB': 'False',
 'CompressedData': 'False',
 'TransformMatrix': '0.999813 -0.00289921 0.0191045 0 0.988769 0.149449 -0.0193232 -0.149421 0.988585',
 'Offset': '-71.985 -53.3146 -55.7472',
 'CenterOfRotation': '0 0 0',
 'AnatomicalOrientation': 'RAI',
 'ElementSpacing': '0.273438 0.273437 2.20001',
 'DimSize': '512 512 47',
 'ElementType': 'MET_SHORT',
 'ElementDataFile': 'Case00.raw'}

In [3]:
case_ids = sorted(
    p.stem for p in DATA_DIR.glob("Case*.mhd") if "segmentation" not in p.stem
)
print(f"{len(case_ids)} cases found")
case_ids[:5]

50 cases found


['Case00', 'Case01', 'Case02', 'Case03', 'Case04']

In [4]:
def direction_to_angle_deg(direction: tuple[float, ...]) -> float:
    """Rotation angle (deg) of the in-plane (x,y) block of the 3x3 direction matrix."""
    d = np.array(direction).reshape(3, 3)
    return float(np.degrees(np.arctan2(d[1, 0], d[0, 0])))


rows = []
for case_id in case_ids:
    mhd_path = DATA_DIR / f"{case_id}.mhd"
    img = sitk.ReadImage(str(mhd_path))
    text_fields = parse_mhd_header(mhd_path)

    spacing = img.GetSpacing()
    dim_size = img.GetSize()
    direction = img.GetDirection()
    origin = img.GetOrigin()

    rows.append(
        {
            "case_id": case_id,
            "dim_x": dim_size[0],
            "dim_y": dim_size[1],
            "dim_z": dim_size[2],
            "spacing_x": round(spacing[0], 4),
            "spacing_y": round(spacing[1], 4),
            "spacing_z": round(spacing[2], 4),
            "direction_angle_deg": round(direction_to_angle_deg(direction), 2),
            "origin_x": round(origin[0], 2),
            "origin_y": round(origin[1], 2),
            "origin_z": round(origin[2], 2),
            "element_type": text_fields.get("ElementType"),
            "anatomical_orientation": text_fields.get("AnatomicalOrientation"),
        }
    )

df = pd.DataFrame(rows)
df

,case_id,dim_x,dim_y,dim_z,spacing_x,spacing_y,spacing_z,direction_angle_deg,origin_x,origin_y,origin_z,element_type,anatomical_orientation
0,Case00,512,512,47,0.2734,0.2734,2.2000,-0.17,-71.98,-53.31,-55.75,MET_SHORT,RAI
1,Case01,512,512,29,0.3516,0.3516,3.0003,0.06,-93.04,-64.04,-138.27,MET_SHORT,RAI
2,Case02,512,512,54,0.2734,0.2734,2.2000,0.00,-68.25,-75.34,-37.34,MET_SHORT,RAI
3,Case03,512,512,42,0.2734,0.2734,2.2000,0.01,-88.90,-55.24,0.06,MET_SHORT,RAI
4,Case04,512,512,46,0.2734,0.2734,2.2005,-0.08,-70.51,-63.28,-120.97,MET_SHORT,RAI
5,Case05,512,512,42,0.2734,0.2734,2.2001,0.00,-67.25,-64.53,-61.88,MET_SHORT,RAI
6,Case06,512,512,39,0.3125,0.3125,2.7999,-0.04,-87.87,-77.14,-48.04,MET_SHORT,RAI
7,Case07,512,512,34,0.2734,0.2734,2.9999,0.05,-53.96,-37.86,-80.35,MET_SHORT,RAI
8,Case08,512,512,40,0.3125,0.3125,2.8000,-0.02,-94.79,-48.87,-63.69,MET_SHORT,RAI
9,Case09,512,512,47,0.2734,0.2734,2.2000,0.00,-63.72,-67.01,-66.14,MET_SHORT,RAI


## Explore for clustering

Group by rounded (spacing_x, spacing_y, spacing_z, dim_z) — these are typically
protocol/scanner-specific and are the most plausible geometric proxy for site.

In [5]:
group_cols = ["spacing_x", "spacing_y", "spacing_z", "dim_z"]
groups = df.groupby(group_cols)["case_id"].apply(list).sort_index()
print(f"{len(groups)} unique (spacing, dim_z) combinations across {len(df)} cases")
for key, cases in groups.items():
    print(key, "->", len(cases), "cases:", cases)

23 unique (spacing, dim_z) combinations across 50 cases
(0.2734, 0.2734, 2.1999, 46) -> 1 cases: ['Case12']
(0.2734, 0.2734, 2.2, 42) -> 1 cases: ['Case03']
(0.2734, 0.2734, 2.2, 47) -> 2 cases: ['Case00', 'Case09']
(0.2734, 0.2734, 2.2, 54) -> 1 cases: ['Case02']
(0.2734, 0.2734, 2.2001, 42) -> 1 cases: ['Case05']
(0.2734, 0.2734, 2.2005, 46) -> 1 cases: ['Case04']
(0.2734, 0.2734, 2.9999, 34) -> 1 cases: ['Case07']
(0.3125, 0.3125, 2.7999, 39) -> 1 cases: ['Case06']
(0.3125, 0.3125, 2.8, 39) -> 1 cases: ['Case10']
(0.3125, 0.3125, 2.8, 40) -> 1 cases: ['Case08']
(0.3125, 0.3125, 2.8, 45) -> 1 cases: ['Case11']
(0.3516, 0.3516, 3.0, 23) -> 1 cases: ['Case33']
(0.3516, 0.3516, 3.0003, 29) -> 1 cases: ['Case01']
(0.3516, 0.3516, 3.3, 26) -> 1 cases: ['Case36']
(0.3906, 0.3906, 3.3, 23) -> 7 cases: ['Case26', 'Case27', 'Case28', 'Case29', 'Case30', 'Case31', 'Case32']
(0.4688, 0.4688, 3.33, 28) -> 1 cases: ['Case34']
(0.6, 0.6, 4.0, 15) -> 1 cases: ['Case13']
(0.6, 0.6, 4.0, 17) -> 1 cas

## KMeans clustering on geometric features

Try clustering into 4 groups (matching the paper's 4 contributing centers) using
spacing, z-extent, and in-plane rotation angle as features. This is exploratory only —
cluster labels (`Site A/B/C/D`) do **not** correspond to verified institution names.

In [6]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

feature_cols = ["spacing_x", "spacing_y", "spacing_z", "dim_z", "direction_angle_deg"]
X = StandardScaler().fit_transform(df[feature_cols])

kmeans = KMeans(n_clusters=4, n_init=10, random_state=0)
df["cluster"] = kmeans.fit_predict(X)
df["site_label"] = df["cluster"].map({0: "Site A", 1: "Site B", 2: "Site C", 3: "Site D"})

print("Cluster sizes:")
print(df["site_label"].value_counts().sort_index())
df[["case_id", *feature_cols, "site_label"]].sort_values("site_label")

Cluster sizes:
site_label
Site A    12
Site B    26
Site C    11
Site D     1
Name: count, dtype: int64


,case_id,spacing_x,spacing_y,spacing_z,dim_z,direction_angle_deg,site_label
26,Case26,0.3906,0.3906,3.3000,23,0.00,Site A
1,Case01,0.3516,0.3516,3.0003,29,0.06,Site A
28,Case28,0.3906,0.3906,3.3000,23,0.00,Site A
29,Case29,0.3906,0.3906,3.3000,23,0.00,Site A
30,Case30,0.3906,0.3906,3.3000,23,0.00,Site A
31,Case31,0.3906,0.3906,3.3000,23,0.00,Site A
32,Case32,0.3906,0.3906,3.3000,23,0.00,Site A
7,Case07,0.2734,0.2734,2.9999,34,0.05,Site A
33,Case33,0.3516,0.3516,3.0000,23,0.00,Site A
34,Case34,0.4688,0.4688,3.3300,28,0.00,Site A


## Per-cluster feature summary

Sanity-check whether the 4 clusters correspond to visibly distinct scanning
protocols (spacing / matrix size / orientation) — supporting the hypothesis that
they reflect different sites, even though we cannot name those sites with confidence.

In [7]:
df.groupby("site_label")[feature_cols].describe().T

site_label                    Site A     Site B     Site C  Site D
spacing_x           count  12.000000  26.000000  11.000000   1.000
                    mean    0.377600   0.630888   0.287618   0.625
                    std     0.045112   0.029714   0.019727     NaN
                    min     0.273400   0.600000   0.273400   0.625
                    25%     0.351600   0.625000   0.273400   0.625
                    50%     0.390600   0.625000   0.273400   0.625
                    75%     0.390600   0.625000   0.312500   0.625
                    max     0.468800   0.750000   0.312500   0.625
spacing_y           count  12.000000  26.000000  11.000000   1.000
                    mean    0.377600   0.630888   0.287618   0.625
                    std     0.045112   0.029714   0.019727     NaN
                    min     0.273400   0.600000   0.273400   0.625
                    25%     0.351600   0.625000   0.273400   0.625
                    50%     0.390600   0.625000   0.273400   0.625
                    75%     0.390600   0.625000   0.312500   0.625
                    max     0.468800   0.750000   0.312500   0.625
spacing_z           count  12.000000  26.000000  11.000000   1.000
                    mean    3.227517   3.634615   2.418218   3.600
                    std     0.137422   0.146812   0.302666     NaN
                    min     2.999900   3.300000   2.199900   3.600
                    25%     3.225075   3.600000   2.200000   3.600
                    50%     3.300000   3.600000   2.200100   3.600
                    75%     3.300000   3.600000   2.799950   3.600
                    max     3.330000   4.000000   2.800000   3.600
dim_z               count  12.000000  26.000000  11.000000   1.000
                    mean   25.083333  21.884615  44.272727  20.000
                    std     3.553701   2.902784   4.474168     NaN
                    min    23.000000  15.000000  39.000000  20.000
                    25%    23.000000  20.000000  41.000000  20.000
                    50%    23.000000  23.500000  45.000000  20.000
                    75%    26.500000  24.000000  46.500000  20.000
                    max    34.000000  28.000000  54.000000  20.000
direction_angle_deg count  12.000000  26.000000  11.000000   1.000
                    mean    0.009167  -0.035000  -0.002727   1.500
                    std     0.021515   0.170488   0.102478     NaN
                    min     0.000000  -0.870000  -0.170000   1.500
                    25%     0.000000   0.000000  -0.030000   1.500
                    50%     0.000000   0.000000   0.000000   1.500
                    75%     0.000000   0.000000   0.005000   1.500
                    max     0.060000  -0.000000   0.260000   1.500